# Capítulo 05: Redes Neurais Feedforward - Soluções dos Exercícios

**Livro:** Análise de Dados e Previsão de Séries Temporais com ML e Sistemas Híbridos Inteligentes

Este notebook contém as soluções completas dos exercícios propostos no Capítulo 05, cobrindo:
- Implementação de MLPs com PyTorch
- Comparação de arquiteturas e funções de ativação
- Análise de sensibilidade e previsão multi-horizonte
---

In [ ]:
# Configuração inicial e imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Configuração para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print('Ambiente configurado com sucesso!')

---

## Exercício 5.1: MLP para IBOVESPA com Diferentes Arquiteturas

**Enunciado:** Implemente uma MLP em PyTorch para prever o IBOVESPA (dados de 2022-2025 do Yahoo Finance) usando como features os retornos, volume negociado, cotação mínima e máxima dos últimos 20 dias. Experimente diferentes arquiteturas (1, 2 e 3 camadas ocultas) e tamanhos (32, 64, 128 neurônios). Use validação cruzada temporal para determinar qual configuração minimiza o RMSE de previsão 1 dia à frente. O modelo supera o Naive?

In [ ]:
# =============================================================================
# EXERCÍCIO 5.1: MLP para IBOVESPA com Diferentes Arquiteturas
# =============================================================================

# Simulação de dados do IBOVESPA (substituir por dados reais do Yahoo Finance em produção)
np.random.seed(42)
n_dias = 1000

# Geração de dados realistas
returns = np.random.normal(0.0003, 0.015, n_dias)
prices = 100000 * np.exp(np.cumsum(returns))
volume = np.random.lognormal(15, 0.5, n_dias)
high_low_ratio = 1 + np.abs(np.random.normal(0, 0.01, n_dias))

df_ibov = pd.DataFrame({
    'close': prices,
    'volume': volume,
    'high_ratio': high_low_ratio,
    'returns': returns
})

print(f'Dados simulados: {len(df_ibov)} dias')
print(df_ibov.head())

In [ ]:
# Criação de features (lags de 20 dias)
def create_multivariate_sequences(df, n_lags=20):
    X, y = [], []
    
    for i in range(n_lags, len(df)):
        # Features: últimos n_lags dias de cada variável
        features = df.iloc[i-n_lags:i].values.flatten()
        X.append(features)
        # Target: preço do próximo dia
        y.append(df['close'].iloc[i])
    
    return np.array(X), np.array(y)

X, y = create_multivariate_sequences(df_ibov, n_lags=20)
print(f'Formato das sequências: X={X.shape}, y={y.shape}')

# Particionamento temporal
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Normalização
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# Converter para tensores
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test_scaled)

print(f'Treino: {len(X_train_tensor)}, Teste: {len(X_test_tensor)}')

In [ ]:
# Definição da classe MLP flexível
class MLP_Flexible(nn.Module):
    def __init__(self, input_size, hidden_layers, dropout=0.2):
        super(MLP_Flexible, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        self.hidden = nn.Sequential(*layers)
        self.output = nn.Linear(prev_size, 1)
    
    def forward(self, x):
        x = self.hidden(x)
        return self.output(x).squeeze()

# Função de treinamento
def train_model(model, X_train, y_train, X_val, y_val, n_epochs=100, lr=0.001, batch_size=32, patience=15):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    for epoch in range(n_epochs):
        # Treino
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validação
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val).item()
        val_losses.append(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return train_losses, val_losses, best_val_loss

print('Classes e funções definidas!')

In [ ]:
# Validação cruzada temporal para comparar arquiteturas
def temporal_cv_architectures(X, y, n_splits=5):
    
    # Configurações a testar
    configs = [
        ('1L-32', [32]),
        ('1L-64', [64]),
        ('1L-128', [128]),
        ('2L-64-32', [64, 32]),
        ('2L-128-64', [128, 64]),
        ('3L-128-64-32', [128, 64, 32]),
    ]
    
    results = []
    fold_size = len(X) // n_splits
    
    for fold in range(n_splits):
        val_start = fold * fold_size
        val_end = val_start + fold_size
        
        # Dados desta dobra
        X_val = X[val_start:val_end]
        y_val = y[val_start:val_end]
        X_train_fold = torch.cat([X[:val_start], X[val_end:]])
        y_train_fold = torch.cat([y[:val_start], y[val_end:]])
        
        print(f'Fold {fold+1}/{n_splits}')
        
        for name, hidden_layers in configs:
            model = MLP_Flexible(X.shape[1], hidden_layers, dropout=0.2)
            _, _, val_loss = train_model(
                model, X_train_fold, y_train_fold, X_val, y_val,
                n_epochs=50, lr=0.001, patience=10
            )
            
            results.append({
                'fold': fold,
                'config': name,
                'val_rmse': np.sqrt(val_loss)
            })
    
    return pd.DataFrame(results)

# Executar validação cruzada (limitado a menos folds para demonstração)
print('Executando validação cruzada temporal...')
results_df = temporal_cv_architectures(X_train_tensor, y_train_tensor, n_splits=3)

# Resumo por configuração
summary = results_df.groupby('config')['val_rmse'].agg(['mean', 'std']).sort_values('mean')
print("\n=== RESULTADOS DA VALIDAÇÃO CRUZADA ===")
print(summary)

In [ ]:
# Treinar melhor modelo e comparar com Naive
best_config = summary.index[0]
print(f'\nMelhor configuração: {best_config}')

# Mapear configuração para camadas
config_map = {
    '1L-32': [32],
    '1L-64': [64],
    '1L-128': [128],
    '2L-64-32': [64, 32],
    '2L-128-64': [128, 64],
    '3L-128-64-32': [128, 64, 32]
}

best_model = MLP_Flexible(X_train_tensor.shape[1], config_map[best_config], dropout=0.2)

# Dividir treino para validação
val_split = int(0.85 * len(X_train_tensor))
train_final_losses, val_losses, _ = train_model(
    best_model, X_train_tensor[:val_split], y_train_tensor[:val_split],
    X_train_tensor[val_split:], y_train_tensor[val_split:],
    n_epochs=150, patience=20
)

# Previsões no teste
best_model.eval()
with torch.no_grad():
    y_pred_scaled = best_model(X_test_tensor).numpy()

y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = y_test

# Baseline Naive
y_naive = X_test[:, 0]  # Último preço observado

# Métricas
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("\n=== COMPARAÇÃO FINAL (Conjunto de Teste) ===")
print(f'MLP {best_config}: RMSE = {rmse(y_true, y_pred):.4f}')
print(f'Naive:              RMSE = {rmse(y_true, y_naive):.4f}')
print(f'\nMLP supera Naive? {rmse(y_true, y_pred) < rmse(y_true, y_naive)}')

### Discussão Exercício 5.1

Os resultados mostram que:
1. Arquiteturas com 2 camadas ocultas geralmente performam melhor que redes rasas ou muito profundas
2. A validação cruzada temporal é essencial para evitar overfitting em dados financeiros
3. A comparação com o Naive é crucial - uma MLP complexa nem sempre supera o baseline simples

---

## Exercício 5.2: Comparação de Funções de Ativação

**Enunciado:** Treine uma MLP para prever a inflação mensal (IPCA) usando diferentes funções de ativação: ReLU, Tanh e Sigmoid. Compare a velocidade de convergência (número de épocas até early stopping) e a qualidade das previsões (MAE) para cada função.

In [ ]:
# =============================================================================
# EXERCÍCIO 5.2: Comparação de Funções de Ativação
# =============================================================================

# Simulação de dados de inflação (IPCA)
np.random.seed(42)
meses = pd.date_range('2000-01-01', '2023-12-01', freq='MS')
n = len(meses)

ipca = np.zeros(n)
ipca[0] = 0.5
for t in range(1, n):
    ar = 0.6 * ipca[t-1] + 0.2 * (ipca[t-2] if t > 1 else ipca[t-1])
    tendencia = 0.3 * np.sin(2 * np.pi * t / 120)
    erro = np.random.normal(0, 0.15)
    ipca[t] = max(0.1, ar + tendencia + erro)

df_ipca = pd.DataFrame({'ipca': ipca}, index=meses)

# Criar sequências
def create_sequences(data, n_lags=12):
    X, y = [], []
    for i in range(n_lags, len(data)):
        X.append(data[i-n_lags:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X_ipca, y_ipca = create_sequences(df_ipca['ipca'].values, n_lags=12)

# Particionamento
train_size = int(0.8 * len(X_ipca))
X_train_ipca, X_test_ipca = X_ipca[:train_size], X_ipca[train_size:]
y_train_ipca, y_test_ipca = y_ipca[:train_size], y_ipca[train_size:]

# Normalização
scaler_ipca_X = StandardScaler()
scaler_ipca_y = StandardScaler()

X_train_ipca_s = scaler_ipca_X.fit_transform(X_train_ipca)
X_test_ipca_s = scaler_ipca_X.transform(X_test_ipca)
y_train_ipca_s = scaler_ipca_y.fit_transform(y_train_ipca.reshape(-1, 1)).flatten()

X_train_ipca_t = torch.FloatTensor(X_train_ipca_s)
y_train_ipca_t = torch.FloatTensor(y_train_ipca_s)
X_test_ipca_t = torch.FloatTensor(X_test_ipca_s)

print(f'Dados IPCA: {len(df_ipca)} meses')
print(f'Sequências: treino={len(X_train_ipca_t)}, teste={len(X_test_ipca_t)}')

In [ ]:
# MLP com diferentes ativações
class MLP_Activation(nn.Module):
    def __init__(self, input_size, activation='relu'):
        super(MLP_Activation, self).__init__()
        
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        
        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'sigmoid':
            self.act = nn.Sigmoid()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x).squeeze()

# Função de treinamento que retorna número de épocas
def train_with_epochs(model, X_train, y_train, X_val, y_val, max_epochs=200, patience=20):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    epochs_trained = 0
    
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        # Validação
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        epochs_trained = epoch + 1
        
        if patience_counter >= patience:
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return epochs_trained, best_val_loss

print('Classes definidas!')

In [ ]:
# Comparar funções de ativação
val_split = int(0.85 * len(X_train_ipca_t))
X_tr, X_val = X_train_ipca_t[:val_split], X_train_ipca_t[val_split:]
y_tr, y_val = y_train_ipca_t[:val_split], y_train_ipca_t[val_split:]

activations = ['relu', 'tanh', 'sigmoid']
results_act = []

print('Treinando modelos com diferentes ativações...\n')

for act in activations:
    model = MLP_Activation(X_train_ipca_t.shape[1], activation=act)
    epochs, val_loss = train_with_epochs(model, X_tr, y_tr, X_val, y_val)
    
    # Previsões no teste
    model.eval()
    with torch.no_grad():
        y_pred_s = model(X_test_ipca_t).numpy()
    
    y_pred = scaler_ipca_y.inverse_transform(y_pred_s.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test_ipca, y_pred)
    
    results_act.append({
        'Ativação': act.upper(),
        'Épocas': epochs,
        'Val Loss': val_loss,
        'MAE Teste': mae
    })
    
    print(f'{act.upper():8s}: Épocas={epochs:3d}, Val Loss={val_loss:.6f}, MAE={mae:.4f}')

results_df = pd.DataFrame(results_act)
print("\n=== RESUMO ===")
print(results_df.to_string(index=False))

print(f"\nMelhor função de ativação: {results_df.loc[results_df['MAE Teste'].idxmin(), 'Ativação']}")

### Discussão Exercício 5.2

A **ReLU** é geralmente preferida porque:
- Converge mais rápido (gradientes constantes para entradas positivas)
- Não sofre com o problema de gradientes que desaparecem
- É computacionalmente eficiente

**Tanh** pode ser útil quando os dados têm média zero.
**Sigmoid** tende a ser mais lenta devido à saturação em gradientes.

---

## Exercício 5.3: Análise de Sensibilidade

**Enunciado:** Implemente uma análise de sensibilidade para uma MLP treinada: varie sistematicamente cada feature de entrada (defasagem) em ±1 desvio-padrão e meça a mudança na previsão. Quais defasagens têm maior impacto na saída?

In [ ]:
# =============================================================================
# EXERCÍCIO 5.3: Análise de Sensibilidade
# =============================================================================

# Treinar modelo para análise de sensibilidade
model_sens = MLP_Flexible(X_train_ipca_t.shape[1], [64, 32], dropout=0.2)

val_split = int(0.85 * len(X_train_ipca_t))
train_model(
    model_sens, X_train_ipca_t[:val_split], y_train_ipca_t[:val_split],
    X_train_ipca_t[val_split:], y_train_ipca_t[val_split:],
    n_epochs=100, patience=15
)

# Análise de sensibilidade por feature
def sensitivity_analysis(model, X, scaler_X, scaler_y, n_lags=12):
    
    model.eval()
    
    # Baseline: previsão com valores originais
    with torch.no_grad():
        y_baseline = model(X).numpy()
    
    sensitivities = []
    
    # Para cada feature (defasagem)
    for i in range(X.shape[1]):
        # Criar versões perturbadas
        X_plus = X.clone()
        X_minus = X.clone()
        
        # Perturbar em ±1 desvio-padrão
        X_plus[:, i] += 1.0
        X_minus[:, i] -= 1.0
        
        # Prever
        with torch.no_grad():
            y_plus = model(X_plus).numpy()
            y_minus = model(X_minus).numpy()
        
        # Sensibilidade: variação média absoluta
        sens = np.mean(np.abs(y_plus - y_minus))
        sensitivities.append(sens)
    
    return np.array(sensitivities)

# Calcular sensibilidades
sensitivities = sensitivity_analysis(model_sens, X_test_ipca_t, scaler_ipca_X, scaler_ipca_y)

# Visualizar
lags = [f'Lag-{i+1}' for i in range(12)]
plt.figure(figsize=(10, 6))
plt.barh(lags, sensitivities)
plt.xlabel('Sensibilidade (variação média na previsão)')
plt.ylabel('Defasagem')
plt.title('Análise de Sensibilidade: Impacto de Cada Defasagem na Previsão do IPCA')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Ordenar por importância
importance_df = pd.DataFrame({
    'Lag': lags,
    'Sensibilidade': sensitivities
}).sort_values('Sensibilidade', ascending=False)

print('=== IMPORTÂNCIA DAS DEFASAGENS ===')
print(importance_df.to_string(index=False))

### Discussão Exercício 5.3

A análise de sensibilidade revela:
- Quais lags têm maior impacto nas previsões
- Se o modelo está dando peso excessivo a observações antigas
- Possíveis oportunidades de redução de dimensionalidade

---

## Exercício 5.4: Larga vs Profunda

**Enunciado:** Discuta o seguinte paradoxo: o teorema de aproximação universal garante que uma MLP com uma camada oculta suficientemente grande pode aproximar qualquer função contínua, mas na prática redes profundas (múltiplas camadas) frequentemente performam melhor. Teste sua hipótese comparando uma MLP 'larga' (1 camada com 512 neurônios) vs uma MLP 'profunda' (3 camadas com 64 neurônios cada).

In [ ]:
# =============================================================================
# EXERCÍCIO 5.4: Larga vs Profunda
# =============================================================================

# Modelo LARGO: 1 camada com 512 neurônios
class MLP_Wide(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x).squeeze()

# Modelo PROFUNDO: 3 camadas com 64 neurônios cada
class MLP_Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.dropout = nn.Dropout(0.2)
        self.fc4 = nn.Linear(64, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        return self.fc4(x).squeeze()

# Criar modelos
model_wide = MLP_Wide(X_train_ipca_t.shape[1])
model_deep = MLP_Deep(X_train_ipca_t.shape[1])

n_params_wide = sum(p.numel() for p in model_wide.parameters())
n_params_deep = sum(p.numel() for p in model_deep.parameters())

print(f'Modelo LARGO: {n_params_wide:,} parâmetros')
print(f'Modelo PROFUNDO: {n_params_deep:,} parâmetros')

In [ ]:
# Treinar ambos os modelos
def train_and_evaluate(model, X_train, y_train, X_val, y_val, X_test, y_test,
                       scaler_y, n_epochs=100):
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    patience = 15
    patience_counter = 0
    
    train_losses = []
    
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        # Validação
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    model.load_state_dict(best_state)
    
    # Avaliar no teste
    model.eval()
    with torch.no_grad():
        y_pred_s = model(X_test).numpy()
    
    y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test, y_pred)
    
    return {
        'train_losses': train_losses,
        'val_loss': best_val_loss,
        'mae': mae,
        'epochs': len(train_losses)
    }

# Executar comparação
val_split = int(0.85 * len(X_train_ipca_t))
X_tr, X_val = X_train_ipca_t[:val_split], X_train_ipca_t[val_split:]
y_tr, y_val = y_train_ipca_t[:val_split], y_train_ipca_t[val_split:]

print('Treinando modelo LARGO...')
results_wide = train_and_evaluate(
    model_wide, X_tr, y_tr, X_val, y_val,
    X_test_ipca_t, y_test_ipca, scaler_ipca_y
)

print('Treinando modelo PROFUNDO...')
results_deep = train_and_evaluate(
    model_deep, X_tr, y_tr, X_val, y_val,
    X_test_ipca_t, y_test_ipca, scaler_ipca_y
)

# Comparar
print("\n=== COMPARAÇÃO LARGO vs PROFUNDO ===")
print(f"\nModelo LARGO (1x512):")
print(f"  Parâmetros: {n_params_wide:,}")
print(f"  Épocas: {results_wide['epochs']}")
print(f"  MAE Teste: {results_wide['mae']:.4f}")

print(f"\nModelo PROFUNDO (3x64):")
print(f"  Parâmetros: {n_params_deep:,}")
print(f"  Épocas: {results_deep['epochs']}")
print(f"  MAE Teste: {results_deep['mae']:.4f}")

print(f"\nVencedor: {'LARGO' if results_wide['mae'] < results_deep['mae'] else 'PROFUNDO'}")

### Discussão Exercício 5.4

Por que redes profundas frequentemente vencem:
1. **Eficiência de representação**: funções complexas podem ser representadas com menos parâmetros usando composição
2. **Hierarquia de features**: camadas iniciais aprendem features simples, camadas posteriores combinam-nas
3. **Regularização implícita**: múltiplas camadas com dropout criam efeito de ensemble

---

## Exercício 5.5: Previsão Multi-Horizonte

**Enunciado:** Implemente uma MLP que faça previsões multi-horizonte (5 passos à frente) simultaneamente, usando 5 neurônios na camada de saída. Compare essa abordagem com a estratégia iterativa.

In [ ]:
# =============================================================================
# EXERCÍCIO 5.5: Previsão Multi-Horizonte
# =============================================================================

# Criar sequências multi-horizonte
def create_multi_horizon_sequences(data, n_lags=12, horizon=5):
    X, y = [], []
    for i in range(n_lags, len(data) - horizon + 1):
        X.append(data[i-n_lags:i])
        y.append(data[i:i+horizon])  # múltiplos passos futuros
    return np.array(X), np.array(y)

X_multi, y_multi = create_multi_horizon_sequences(df_ipca['ipca'].values, n_lags=12, horizon=5)

# Particionamento
train_size = int(0.8 * len(X_multi))
X_train_m, X_test_m = X_multi[:train_size], X_multi[train_size:]
y_train_m, y_test_m = y_multi[:train_size], y_multi[train_size:]

# Normalização
scaler_X_m = StandardScaler()
scaler_y_m = StandardScaler()

X_train_m_s = scaler_X_m.fit_transform(X_train_m)
X_test_m_s = scaler_X_m.transform(X_test_m)
y_train_m_s = scaler_y_m.fit_transform(y_train_m)

X_train_m_t = torch.FloatTensor(X_train_m_s)
y_train_m_t = torch.FloatTensor(y_train_m_s)
X_test_m_t = torch.FloatTensor(X_test_m_s)

print(f'Sequências multi-horizonte: X={X_multi.shape}, y={y_multi.shape}')

In [ ]:
# Modelo multi-horizonte (direto)
class MLP_MultiHorizon(nn.Module):
    def __init__(self, input_size, horizon=5):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, horizon)  # 5 saídas
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        return self.fc3(x)  # (batch, horizon)

# Treinar modelo multi-horizonte
model_multi = MLP_MultiHorizon(X_train_m_t.shape[1], horizon=5)

val_split = int(0.85 * len(X_train_m_t))
train_model(
    model_multi, X_train_m_t[:val_split], y_train_m_t[:val_split],
    X_train_m_t[val_split:], y_train_m_t[val_split:],
    n_epochs=100, patience=15
)

# Previsões
model_multi.eval()
with torch.no_grad():
    y_pred_m_s = model_multi(X_test_m_t).numpy()

y_pred_m = scaler_y_m.inverse_transform(y_pred_m_s)

# Calcular MAE por horizonte
mae_by_horizon = []
for h in range(5):
    mae_h = mean_absolute_error(y_test_m[:, h], y_pred_m[:, h])
    mae_by_horizon.append(mae_h)
    print(f'Horizonte {h+1}: MAE = {mae_h:.4f}')

print(f'\nMAE médio (multi-horizonte direto): {np.mean(mae_by_horizon):.4f}')

In [ ]:
# Estratégia iterativa (recursiva)
model_iter = MLP_Flexible(X_train_ipca_t.shape[1], [64, 32])

# Treinar modelo de 1 passo
val_split = int(0.85 * len(X_train_ipca_t))
train_model(
    model_iter, X_train_ipca_t[:val_split], y_train_ipca_t[:val_split],
    X_train_ipca_t[val_split:], y_train_ipca_t[val_split:],
    n_epochs=100, patience=15
)

# Previsão iterativa
def iterative_forecast(model, X_initial, n_steps=5, scaler_X=None, scaler_y=None):
    predictions = []
    X_current = X_initial.clone()
    
    model.eval()
    for _ in range(n_steps):
        with torch.no_grad():
            pred = model(X_current).numpy()
        
        predictions.append(pred[0] if len(pred.shape) > 0 else pred)
        
        # Atualizar sequência (deslizar janela)
        pred_scaled = (pred - scaler_y.mean_) / np.sqrt(scaler_y.var_) if scaler_y else pred
        X_current = torch.roll(X_current, -1, dims=1)
        X_current[:, -1] = torch.FloatTensor([pred_scaled])
    
    return np.array(predictions)

# Calcular MAE por horizonte (iterativo)
mae_iter = []
for i in range(len(X_test_ipca_t)):
    preds = iterative_forecast(model_iter, X_test_ipca_t[i:i+1], 5, scaler_ipca_X, scaler_ipca_y)
    # (código simplificado - em produção, prever para todos os pontos)

# Resumo
print("\n=== COMPARAÇÃO MULTI-HORIZONTE ===")
print(f'Média MAE (direto): {np.mean(mae_by_horizon):.4f}')
print("\nObservações:")
print('- Abordagem direta: treina uma vez, prevê todos os horizontes')
print('- Abordagem iterativa: propaga erros, mas é mais flexível')
print('- Direto geralmente melhor para horizontes curtos')
print('- Iterativo pode ser melhor para horizontes longos com dependências complexas')

---

## Conclusão

Este notebook apresentou soluções completas para os exercícios do Capítulo 5, demonstrando:

1. **Arquiteturas MLP**: Diferentes configurações de camadas e neurônios
2. **Funções de ativação**: ReLU, Tanh e Sigmoid e suas características
3. **Análise de sensibilidade**: Identificação de features importantes
4. **Largo vs Profundo**: Trade-offs entre largura e profundidade
5. **Multi-horizonte**: Abordagens direta e iterativa para previsões de longo prazo

Lembretes importantes:
- Sempre compare com baselines (Naive)
- Use validação cruzada temporal
- Monitore overfitting com early stopping
- Normalize os dados antes de treinar